In [1]:
import pandas as pd
from tqdm import tqdm

In [2]:
TEST_DATA_PATH = '/home/administrateur/Bureau/Datagrosyst/catalogue_script_agrosyst/02_outils/tests/data/test_get_espece_variete_perenne_principale/'
ENTREPOT_PATH = '/home/administrateur/Bureau/Datagrosyst/data_entrepot_outils/'
donnees = {}

In [3]:
def import_df(df_name, path_data, sep, index_col=None):
    donnees[df_name] = pd.read_csv(path_data+df_name+'.csv', sep = sep, index_col=index_col, low_memory=False).replace({'\r\n': '\n'}, regex=True)

def import_dfs(df_names, path_data, sep = ',', index_col=None, verbose=False):
    for df_name in tqdm(df_names) : 
        if(verbose) :
            print(" - ", df_name)
        import_df(df_name, path_data, sep, index_col=index_col)

tables = [
    'sdc',
    'synthetise',
    'plantation_perenne_synthetise',
    'plantation_perenne_realise',
    'parcelle',
    'zone',
    'espece',
    'variete',
    'composant_culture',
    'plantation_perenne_synthetise_restructure'
    # Si on veut attraper les données des assolés
    # 'noeuds_synthetise_restructure',
    # 'noeuds_synthetise',
    # 'noeuds_realise',
    # 'poids_connexions_synthetise_rotation'
]

# import des données du magasin
import_dfs(tables, ENTREPOT_PATH, sep = ',', verbose=False)

100%|██████████| 10/10 [00:09<00:00,  1.11it/s]


In [4]:
# Création des variables
sdc = donnees['sdc'][['id','filiere']].copy()
synthetise = donnees['synthetise'][['id','sdc_id']].copy()
plantation_perenne_synthetise = donnees['plantation_perenne_synthetise'][['id','synthetise_id','pct_occupation_sol']].copy()
plantation_perenne_realise = donnees['plantation_perenne_realise'][['id','culture_id','zone_id']].copy()
parcelle = donnees['parcelle'][['id','sdc_id']].copy()
zone = donnees['zone'][['id','parcelle_id','surface']].copy()
espece = donnees['espece'][['id','libelle_espece_botanique','typocan_espece']].copy()
variete = donnees['variete'][['id','denomination']].copy()
composant_culture = donnees['composant_culture'].copy()
plantation_perenne_synthetise_restructure = donnees['plantation_perenne_synthetise_restructure'].copy()

# Si on veut attraper les données des assolés
# noeuds_synthetise = donnees['noeuds_synthetise'][['id','synthetise_id']].copy()
# poids_connexions_synthetise_rotation = donnees['poids_connexions_synthetise_rotation'][['connexion_id','poids_conx_agregation']].copy()
# noeuds_realise = donnees['noeuds_realise'][['id','culture_id','zone_id']].copy()
# noeuds_synthetise_restructure = donnees['noeuds_synthetise_restructure'].copy()

In [5]:
gs_souhaite = ["fr.inra.agrosyst.api.entities.GrowingSystem_00e0ed65-a1f8-48ae-b1f5-ecb703099dd7",
"fr.inra.agrosyst.api.entities.GrowingSystem_2009e18f-9e69-4485-b8bd-82a3f1bcf166",
"fr.inra.agrosyst.api.entities.GrowingSystem_1f7f2dc5-b05c-47ec-bb0e-271c80bdc19c",
"fr.inra.agrosyst.api.entities.GrowingSystem_253b46ab-8248-4132-9deb-b5b14bbf62c0",
"fr.inra.agrosyst.api.entities.GrowingSystem_28323105-d6b3-455d-b35c-7588a4e181b3",
"fr.inra.agrosyst.api.entities.GrowingSystem_2b644d06-0b32-44c0-a65c-05102f8911b2",
"fr.inra.agrosyst.api.entities.GrowingSystem_324a736c-060a-4d4f-9c57-e9d731df3431",
"fr.inra.agrosyst.api.entities.GrowingSystem_8b67367c-0bab-420f-917b-2ce5d6ea4d8b"]

sy_souhaite = ["fr.inra.agrosyst.api.entities.practiced.PracticedSystem_34b40970-9d45-47a7-9bf9-6406341c3cb2",
"fr.inra.agrosyst.api.entities.practiced.PracticedSystem_3803d365-e819-4733-a6b4-51465191337c",
"fr.inra.agrosyst.api.entities.practiced.PracticedSystem_f8b0fa56-a504-4fc4-b98d-d3f9cc93e950",
"fr.inra.agrosyst.api.entities.practiced.PracticedSystem_0e46b493-8fcf-4a00-ac8b-57385aaeb6c9",
"fr.inra.agrosyst.api.entities.practiced.PracticedSystem_0c846b24-0e7b-421c-af72-b96d0e199ae9",
"fr.inra.agrosyst.api.entities.practiced.PracticedSystem_119179b1-3237-4382-b271-fe13bca44411",
"fr.inra.agrosyst.api.entities.practiced.PracticedSystem_14883d32-c91a-4477-9dd4-9555be09295d",
"fr.inra.agrosyst.api.entities.practiced.PracticedSystem_20162792-1cf0-4634-8cc0-27356cee9a49",
"fr.inra.agrosyst.api.entities.practiced.PracticedSystem_ecf69a40-2a3c-4f9c-8d9d-07f514c52213",
"fr.inra.agrosyst.api.entities.practiced.PracticedSystem_ae04198b-791b-4be2-aafc-deb7809de537"]

synthetise = donnees['synthetise'].loc[donnees['synthetise']['id'].isin(sy_souhaite)].copy()
sdc = donnees['sdc'].loc[donnees['sdc']['id'].isin(gs_souhaite + list(synthetise['sdc_id']))].copy()
parcelle = donnees['parcelle'].loc[donnees['parcelle']['sdc_id'].isin(sdc['id'])].copy()
zone = donnees['zone'].loc[donnees['zone']['parcelle_id'].isin(parcelle['id'])].copy()
plantation_perenne_realise = donnees['plantation_perenne_realise'].loc[donnees['plantation_perenne_realise']['zone_id'].isin(zone['id'])].copy()
plantation_perenne_synthetise = donnees['plantation_perenne_synthetise'].loc[donnees['plantation_perenne_synthetise']['synthetise_id'].isin(synthetise['id'])].copy()
plantation_perenne_synthetise_restructure = donnees['plantation_perenne_synthetise_restructure'].loc[donnees['plantation_perenne_synthetise_restructure']['id'].isin(plantation_perenne_synthetise['id'])].copy()
espece = donnees['espece'].copy()
variete = donnees['variete'].copy()
composant_culture = donnees['composant_culture'].loc[donnees['composant_culture']['culture_id'].isin(
    list(plantation_perenne_realise['culture_id'].unique()) + 
    list(plantation_perenne_synthetise_restructure['culture_id'].unique())
    )].copy()

In [6]:
sdc.to_csv(TEST_DATA_PATH+'sdc.csv', index=False)
synthetise.to_csv(TEST_DATA_PATH+'synthetise.csv', index=False)
parcelle.to_csv(TEST_DATA_PATH+'parcelle.csv', index=False)
zone.to_csv(TEST_DATA_PATH+'zone.csv', index=False)
plantation_perenne_realise.to_csv(TEST_DATA_PATH+'plantation_perenne_realise.csv', index=False)
plantation_perenne_synthetise.to_csv(TEST_DATA_PATH+'plantation_perenne_synthetise.csv', index=False)
plantation_perenne_synthetise_restructure.to_csv(TEST_DATA_PATH+'plantation_perenne_synthetise_restructure.csv', index=False)
espece.to_csv(TEST_DATA_PATH+'espece.csv', index=False)
variete.to_csv(TEST_DATA_PATH+'variete.csv', index=False)
composant_culture.to_csv(TEST_DATA_PATH+'composant_culture.csv', index=False)

del(sdc, synthetise, parcelle, zone, plantation_perenne_realise, plantation_perenne_synthetise, plantation_perenne_synthetise_restructure, espece, variete, composant_culture)

In [7]:
# On reimporte les données mais depuis le dossier de test cette fois
import_dfs(tables, TEST_DATA_PATH, sep = ',', verbose=False)

# Création des variables
sdc = donnees['sdc'][['id','filiere']].copy()
synthetise = donnees['synthetise'][['id','sdc_id']].copy()
plantation_perenne_synthetise = donnees['plantation_perenne_synthetise'][['id','synthetise_id','pct_occupation_sol']].copy()
plantation_perenne_realise = donnees['plantation_perenne_realise'][['id','culture_id','zone_id']].copy()
parcelle = donnees['parcelle'][['id','sdc_id']].copy()
zone = donnees['zone'][['id','parcelle_id','surface']].copy()
espece = donnees['espece'][['id','libelle_espece_botanique','typocan_espece']].copy()
variete = donnees['variete'][['id','denomination']].copy()
composant_culture = donnees['composant_culture'].copy()
plantation_perenne_synthetise_restructure = donnees['plantation_perenne_synthetise_restructure'].copy()

100%|██████████| 10/10 [00:00<00:00, 32.32it/s]


In [8]:
# Restructuraion des données
plantation_perenne_synthetise = plantation_perenne_synthetise.merge(plantation_perenne_synthetise_restructure, on='id', how='left')
plantation_perenne_realise = plantation_perenne_realise.merge(zone.rename(columns={'id': 'zone_id'}), on='zone_id', how='left').merge(parcelle.rename(columns={'id': 'parcelle_id'}), on='parcelle_id', how='left')

composant_culture = composant_culture.merge(espece.rename(columns={'id': 'espece_id'}), on='espece_id', how='left').merge(variete.rename(columns={'id': 'variete_id'}), on='variete_id', how='left').rename(columns={'id': 'composant_id'})

# Si on veut attraper les données des assolés
# noeuds_synthetise = noeuds_synthetise.merge(noeuds_synthetise_restructure, on='id', how='left')
# noeuds_realise = noeuds_realise.merge(zone.rename(columns={'id': 'zone_id'}), on='zone_id', how='left').merge(parcelle.rename(columns={'id': 'parcelle_id'}), on='parcelle_id', how='left')


# Concaténation des synthé/réalisé/pérenne
all_peren= pd.concat([
    # noeuds_realise[['id','culture_id','sdc_id']].assign(type='assole'),
    # noeuds_synthetise[['id','culture_id','synthetise_id']].assign(type='assole'),
    plantation_perenne_realise[['id','culture_id','sdc_id','surface']],#.assign(type='peren'),
    plantation_perenne_synthetise[['id','culture_id','synthetise_id','pct_occupation_sol']]#.assign(type='peren')
]).rename(columns={'id': 'perenne_id'})

# Ajout des infos
all_peren['entite_id'] = all_peren['sdc_id'].fillna(all_peren['synthetise_id'])
all_peren = all_peren.merge(synthetise[['id', 'sdc_id']].rename(columns={'id': 'synthetise_id', 'sdc_id': 'sdc_id_fromsynth'}), on='synthetise_id', how='left')
all_peren['sdc_id'] = all_peren['sdc_id'].fillna(all_peren['sdc_id_fromsynth'])
all_peren = all_peren.merge(sdc[['id', 'filiere']].rename(columns={'id': 'sdc_id'}), on='sdc_id', how='left').drop(columns=['sdc_id_fromsynth'], errors='ignore')

# On ne garde que les filiere ARBO et VITI
all_peren = all_peren.loc[all_peren['filiere'].isin(['ARBORICULTURE','VITICULTURE'])]

# On epxlose all_peren pour avoir toutes les composant de culture associé à chaque culture de perenne
composant_culture = composant_culture[['composant_id','culture_id','typocan_espece','denomination','surface_relative']]
df = all_peren.merge(composant_culture, on='culture_id', how='left')
df['pct_occupation_sol'] = df['pct_occupation_sol'].fillna(100) / 100
df['surface_relative'] = df['surface_relative'].fillna(100) / 100
df['surface'] = df['surface'].fillna(1)

# Calcul de la proportion pour les zones
# Si la culture a plusieurs composant par perenne_id, on multiplie la zone par le ratio entre la sruface relative et la somme des surface relative des composants d'une même perenne_id. Sachant que les surface relatives null ont été remplacées par 1.
df["surface_corrigee"] = df["surface"] * (df['surface_relative'] / df.groupby("perenne_id")["surface_relative"].transform("sum"))
# PLusieurs culture_id dans la même zone => zone devellopé. On donne simplement la surface de la zone à la culture
df["surface_prop"] = df["surface_corrigee"] / df.groupby("entite_id")["surface_corrigee"].transform("sum")

# Calcul de la proportion finale au seins de l'entité
df['proportion_composant'] = df['surface_prop'] * df['pct_occupation_sol']


# Verif qu'il y a bien une entite_id renseigné
df = df[df['entite_id'].notna()]

df = df[['perenne_id','entite_id','filiere','typocan_espece','denomination','proportion_composant']]

In [9]:
def specify_colnames_of_return(esp_principale, var_principale, list_esp, list_var):
    """
    Petite fonction pour spécifier les noms de colonnes du return de la fonction get_major_species_and_variety() qui est utilisé dans un groupby.
    """
    return pd.Series({
        'espece_principale': esp_principale,
        'variete_principale': var_principale,
        'liste_toutes_especes': list_esp,
        'liste_toutes_varietes': list_var
    })

def get_major_species_and_variety(groupe, typo_sp_arbo, pct_ecart=2):
    """
    A ECRIRE
    """
    
    # Check qu'il n'y a qu'une seule filière par entite_id
    if groupe["filiere"].nunique() != 1:
        raise ValueError(f"Filières mixtes pour entite_id {groupe.name}")
    filiere = groupe["filiere"].iloc[0]
    if filiere not in ["ARBORICULTURE", "VITICULTURE"]:
        raise ValueError(f"Filière non supportée pour entite_id {groupe.name}")

    # Check qu'il y ait au moins une espèce renseigné
    if groupe["typocan_espece"].nunique() == 0:
        return specify_colnames_of_return('Erreur aucune espèce renseigné', None, None, None)

    # liste de toutes les typologies d'espèce différentes
    list_esp = groupe.groupby("typocan_espece", as_index=False)["proportion_composant"].sum()
    list_esp = list_esp.sort_values("proportion_composant", ascending=False).reset_index(drop=True)
    list_esp = list_esp['typocan_espece'].to_list()

    # liste de toutes les denomination de variété différentes
    list_var = groupe.groupby("denomination", as_index=False)["proportion_composant"].sum()
    list_var = list_var.sort_values("proportion_composant", ascending=False).reset_index(drop=True)
    list_var = list_var['denomination'].to_list()
    
    if filiere == "ARBORICULTURE":
        # a partir de là on ne regarde plus que les composants présent dans la liste des espèce arboricole majoritaire dans DEPHY
        grp_filt = groupe[groupe['typocan_espece'].isin(typo_sp_arbo)]
        if grp_filt.empty : 
            return specify_colnames_of_return('Erreur aucune des espèces arboricoles majoritaires', None, None, None)

        # On refait un groupby pour avoir le classement par proportion des différentes espèces
        grp_by_typo = grp_filt.groupby("typocan_espece", as_index=False)["proportion_composant"].sum()
        grp_by_typo = grp_by_typo.sort_values("proportion_composant", ascending=False).reset_index(drop=True)
        esp_top1 = grp_by_typo.iloc[0]
        esp_principale = esp_top1['typocan_espece']

        # Vérification que le top1 soit bien au dessus du top2 en terme de proportion
        if len(grp_by_typo) > 1 :
            if esp_top1['proportion_composant'] > 0:
                esp_top2 = grp_by_typo.iloc[1]
                pct_diff_esp = (esp_top1['proportion_composant'] - esp_top2['proportion_composant']) / esp_top1['proportion_composant'] * 100
                if pct_diff_esp < pct_ecart : esp_principale = 'Mélange égal d_espèce'
            else : esp_principale = 'Mélange égal d_espèce'
    
    elif filiere == "VITICULTURE":
        # a partir de là on ne regarde plus que les composants présent dans la liste des espèce arboricole majoritaire dans DEPHY
        grp_filt = groupe[groupe['typocan_espece']=='Vigne']
        if grp_filt.empty : 
            return specify_colnames_of_return('Erreur aucune vigne', None, None, None)

        # Pour la viti, seule l'espèce vigne est considéré comme espèce princiaple
        esp_principale = 'Vigne'

    # On check s'il y a des variétés tout court
    if groupe["denomination"].nunique() == 0 : 
            return specify_colnames_of_return(esp_principale, 'Aucune variété renseignée', list_esp, list_var)
    
    # On récupère la variété principale AU SEINS de l'espèce principale, selon la même procédure
    if esp_principale != 'Mélange égal d_espèce' and esp_principale:
        grp_by_denom = grp_filt.loc[(grp_filt['typocan_espece'] == esp_principale) & (grp_filt['denomination'].notna())]
        # Cas où l'on a bien une espèce principale mais aucune varitété de renseignée
        if grp_by_denom.empty : 
            return specify_colnames_of_return(esp_principale, 'Aucune variété pour l_espèce principale', list_esp, list_var)

        grp_by_denom = grp_by_denom.groupby("denomination", as_index=False)["proportion_composant"].sum()
        grp_by_denom = grp_by_denom.sort_values("proportion_composant", ascending=False).reset_index(drop=True)
        var_top1 = grp_by_denom.iloc[0]
        var_principale = var_top1['denomination']

        # Vérification que le top1 soit bien au dessus du top2 en terme de proportion
        if len(grp_by_denom) > 1 :
            if var_top1['proportion_composant'] > 0:
                var_top2 = grp_by_denom.iloc[1]
                pct_diff_var = (var_top1['proportion_composant'] - var_top2['proportion_composant']) / var_top1['proportion_composant'] * 100
                if pct_diff_var < pct_ecart : var_principale = 'Mélange égal de variété'
            else : var_principale = 'Mélange égal de variété'
    else : var_principale = None
    
    return specify_colnames_of_return(esp_principale, var_principale, list_esp, list_var)
    

### Utilisation de la fonction
# Parametres
pct_ecart = 2       # en pourcentage de différence entre le top1 et le top2, 0 à 100
typo_sp_arbo = ["Pommier", "Poirier", "Pêcher", "Abricotier", "Prunier", 
                "Clémentinier", "Noyer", "Olivier", "Cerisier"] 
# code_sp_arbo = ["G21", "G20", "G07", "E01", "G28", "E85", "F84", "F86", "E67"]
# Eventuel à rajouter pour culture trop : ["Ananas", "Bananier plantain", "Manguier"]

# utilisation de la fonction
df_final = df.groupby("entite_id").apply(get_major_species_and_variety, 
                                        typo_sp_arbo=typo_sp_arbo, 
                                        pct_ecart=pct_ecart,
                                        include_groups=False)
df_final

,espece_principale,variete_principale,liste_toutes_especes,liste_toutes_varietes
entite_id,,,,
fr.inra.agrosyst.api.entities.GrowingSystem_00e0ed65-a1f8-48ae-b1f5-ecb703099dd7,Vigne,Merlot N,"[Autre, Protéagineux, Vigne]",[Merlot N]
fr.inra.agrosyst.api.entities.GrowingSystem_1d4a9d61-1eb5-40f7-aa59-6d15bc82af3c,Vigne,Alval,[Vigne],"[Alval, Muscat à petits grains blancs B]"
fr.inra.agrosyst.api.entities.GrowingSystem_1f7f2dc5-b05c-47ec-bb0e-271c80bdc19c,Pêcher,Aucune variété renseignée,[Pêcher],[]
fr.inra.agrosyst.api.entities.GrowingSystem_2009e18f-9e69-4485-b8bd-82a3f1bcf166,Erreur aucune des espèces arboricoles majorita...,None,None,None
fr.inra.agrosyst.api.entities.GrowingSystem_253b46ab-8248-4132-9deb-b5b14bbf62c0,Vigne,Mélange égal de variété,[Vigne],"[Cabernet franc N, Cabernet-Sauvignon N, Cot N..."
fr.inra.agrosyst.api.entities.GrowingSystem_28323105-d6b3-455d-b35c-7588a4e181b3,Pommier,Aucune variété renseignée,[Pommier],[]
fr.inra.agrosyst.api.entities.GrowingSystem_2b644d06-0b32-44c0-a65c-05102f8911b2,Erreur aucune des espèces arboricoles majorita...,None,None,None
fr.inra.agrosyst.api.entities.GrowingSystem_324a736c-060a-4d4f-9c57-e9d731df3431,Erreur aucune espèce renseigné,None,None,None
fr.inra.agrosyst.api.entities.GrowingSystem_8b67367c-0bab-420f-917b-2ce5d6ea4d8b,Pommier,Aucune variété pour l_espèce principale,"[Autre, Pommier, Olivier, Abricotier, Kiwi, Pê...","[Picholine, Tom Cot]"


In [10]:
df_final.to_csv('/home/administrateur/Bureau/resultat_get_espece_variete_principale_perenne.csv', index=True)